In [3]:

# Then import the required libraries
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import pandas as pd

In [4]:
PROJECT_ROOT = os.path.abspath("..")

MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "efficientnet_skin.pth")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

print("MODEL_PATH exists:", os.path.exists(MODEL_PATH))

MODEL_PATH exists: True


In [5]:
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [6]:
# -----------------------------
# Class Definitions
# -----------------------------
CLASS_CODES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

CLASS_LABELS = {
    "akiec": "Actinic Keratoses and Intraepithelial Carcinoma",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevus",
    "vasc": "Vascular Lesion"
}


In [7]:
# -----------------------------
# Load EfficientNet-B0
# -----------------------------
model = models.efficientnet_b0()
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 7)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("EfficientNet loaded successfully.")

EfficientNet loaded successfully.


In [9]:
# Image Transform & Prediction

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image)
        probs = F.softmax(logits, dim=1)

    confidence, predicted_class = torch.max(probs, dim=1)

    return {
        "predicted_index": predicted_class.item(),
        "predicted_disease": CLASS_LABELS[CLASS_CODES[predicted_class.item()]],
        "confidence": confidence.item(),
        "probabilities": probs.squeeze().tolist()
    }

In [11]:
# Confidence Decision Logic

HIGH_CONF_THRESHOLD = 0.80
LOW_CONF_THRESHOLD = 0.40

def confidence_decision(confidence):
    if confidence >= HIGH_CONF_THRESHOLD:
        return "SAFE_ADVISORY"
    elif confidence >= LOW_CONF_THRESHOLD:
        return "ASK_MORE_QUESTIONS"
    else:
        return "ETHICAL_REFUSAL"

def image_pre_diagnosis(image_path):
    result = predict_image(image_path)

    decision = confidence_decision(result["confidence"])

    return {
        "predicted_disease": result["predicted_disease"],
        "confidence": round(result["confidence"], 4),
        "decision": decision,
        "probabilities": result["probabilities"]
    }

In [18]:
# Load Medical Knowledge Base
symptom_questions = pd.read_csv(
    os.path.join(DATA_DIR, "symptom_questions.csv")
)

disease_symptom_matrix = pd.read_csv(
    os.path.join(DATA_DIR, "disease_symptom_matrix.csv")
)

question_priority = pd.read_csv(
    os.path.join(DATA_DIR, "question_priority.csv")
)

print("Medical reasoning datasets loaded.")

Medical reasoning datasets loaded.


In [17]:
#Symptom Mapping Functions
def get_symptoms_for_disease(disease):

    row = disease_symptom_matrix[
        disease_symptom_matrix["disease"] == disease
    ]

    if row.empty:
        return []

    symptom_row = row.drop(columns=["disease"]).iloc[0]
    symptom_ids = symptom_row[symptom_row == 1].index.tolist()

    return symptom_ids


def get_questions_for_symptoms(symptom_ids):
    return symptom_questions[
        symptom_questions["symptom_id"].isin(symptom_ids)
    ]


def prioritize_questions(questions_df):
    merged = questions_df.merge(
        question_priority,
        on="symptom_id",
        how="left"
    )

    return merged.sort_values(
        by="priority_weight",
        ascending=False
    )

In [16]:
#Adaptive Questioning
def ask_questions(questions_df, max_questions=3):

    answers = {}

    for _, row in questions_df.head(max_questions).iterrows():

        question = row["question"]
        symptom_id = row["symptom_id"]

        # Simulated input (replace later with real input)
        simulated_answer = "yes"
        answers[symptom_id] = simulated_answer

        print(f"Q: {question}")
        print(f"A: {simulated_answer}")
        print("-" * 40)

    return answers


def score_answers(answers):

    score_map = {
        "no": 0.0,
        "mild": 0.5,
        "yes": 1.0,
        "severe": 1.0
    }

    total_score = 0.0
    for ans in answers.values():
        total_score += score_map.get(ans.lower(), 0.0)

    return total_score / max(len(answers), 1)

In [19]:
#Adaptive Reasoning Pipeline

def adaptive_questioning_pipeline(image_output):

    if image_output["decision"] != "ASK_MORE_QUESTIONS":
        return None

    disease = image_output["predicted_disease"]

    symptom_ids = get_symptoms_for_disease(disease)
    questions = get_questions_for_symptoms(symptom_ids)
    ordered_questions = prioritize_questions(questions)

    answers = ask_questions(ordered_questions)
    symptom_score = score_answers(answers)

    return {
        "symptom_score": round(symptom_score, 4),
        "answers": answers
    }

In [20]:
#Probabilistic Fusion Logic

FUSION_HIGH_CONF = 0.80
FUSION_MED_CONF = 0.55
SYMPTOM_SUPPORT_HIGH = 0.60

def final_decision_fusion(image_output, reasoning_output=None):

    confidence = image_output["confidence"]

    if confidence < LOW_CONF_THRESHOLD:
        return "ETHICAL_REFUSAL"

    if reasoning_output is None:
        if confidence >= FUSION_HIGH_CONF:
            return "SAFE_ADVISORY"
        else:
            return "LOW_CONFIDENCE_WARNING"

    symptom_score = reasoning_output["symptom_score"]

    if confidence >= FUSION_HIGH_CONF and symptom_score >= SYMPTOM_SUPPORT_HIGH:
        return "SAFE_ADVISORY"

    if confidence >= FUSION_MED_CONF or symptom_score >= SYMPTOM_SUPPORT_HIGH:
        return "LOW_CONFIDENCE_WARNING"

    return "ETHICAL_REFUSAL"

In [21]:
# Full Pre-Diagnosis Pipeline

def full_pre_diagnosis(image_path):

    image_output = image_pre_diagnosis(image_path)

    reasoning_output = adaptive_questioning_pipeline(image_output)

    final_decision = final_decision_fusion(
        image_output,
        reasoning_output
    )

    return {
        "image_output": image_output,
        "reasoning_output": reasoning_output,
        "final_decision": final_decision
    }

In [22]:
test_image_path = os.path.join(
    DATA_DIR,
    "HAM10000_images_part_1",
    os.listdir(os.path.join(DATA_DIR, "HAM10000_images_part_1"))[0]
)

final_output = full_pre_diagnosis(test_image_path)
final_output

{'image_output': {'predicted_disease': 'Melanocytic Nevus',
  'confidence': 0.7824,
  'decision': 'ASK_MORE_QUESTIONS',
  'probabilities': [0.005170303396880627,
   0.009318500757217407,
   0.10561399161815643,
   0.03289561718702316,
   0.03132232278585434,
   0.7823680639266968,
   0.03331124410033226]},
 'reasoning_output': {'symptom_score': 0.0, 'answers': {}},
 'final_decision': 'LOW_CONFIDENCE_WARNING'}

In [24]:
# ============================================================
# EfficientNet-Based Confidence-Aware Medical Pre-Diagnosis
# Bayesian Posterior Update (Improved Version)
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
PROJECT_ROOT = os.path.abspath("..")
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "efficientnet_skin.pth")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

print("Model exists:", os.path.exists(MODEL_PATH))

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# CLASS DEFINITIONS
# ------------------------------------------------------------
CLASS_CODES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

CLASS_LABELS = {
    "akiec": "Actinic Keratoses and Intraepithelial Carcinoma",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevus",
    "vasc": "Vascular Lesion"
}

# ------------------------------------------------------------
# LOAD EFFICIENTNET
# ------------------------------------------------------------
model = models.efficientnet_b0()
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 7)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("EfficientNet loaded successfully.")

# ------------------------------------------------------------
# IMAGE TRANSFORM
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

# ------------------------------------------------------------
# IMAGE PREDICTION
# ------------------------------------------------------------
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image)
        probs = F.softmax(logits, dim=1)

    confidence, predicted_class = torch.max(probs, dim=1)

    return {
        "predicted_index": predicted_class.item(),
        "predicted_disease": CLASS_LABELS[CLASS_CODES[predicted_class.item()]],
        "confidence": confidence.item(),
        "probabilities": probs.squeeze().tolist()
    }

# ------------------------------------------------------------
# CONFIDENCE THRESHOLDS (UPDATED)
# ------------------------------------------------------------
HIGH_CONF_THRESHOLD = 0.75   # Reduced from 0.80
LOW_CONF_THRESHOLD = 0.40

def confidence_decision(confidence):
    if confidence >= HIGH_CONF_THRESHOLD:
        return "SAFE_ADVISORY"
    elif confidence >= LOW_CONF_THRESHOLD:
        return "ASK_MORE_QUESTIONS"
    else:
        return "ETHICAL_REFUSAL"

def image_pre_diagnosis(image_path):
    result = predict_image(image_path)
    decision = confidence_decision(result["confidence"])

    return {
        "predicted_disease": result["predicted_disease"],
        "confidence": result["confidence"],
        "decision": decision,
        "probabilities": result["probabilities"]
    }

# ------------------------------------------------------------
# LOAD MEDICAL KNOWLEDGE BASE
# ------------------------------------------------------------
symptom_questions = pd.read_csv(
    os.path.join(DATA_DIR, "symptom_questions.csv")
)

disease_symptom_matrix = pd.read_csv(
    os.path.join(DATA_DIR, "disease_symptom_matrix.csv")
)

question_priority = pd.read_csv(
    os.path.join(DATA_DIR, "question_priority.csv")
)

print("Medical datasets loaded.")

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def get_symptoms_for_disease(disease):
    row = disease_symptom_matrix[
        disease_symptom_matrix["disease"] == disease
    ]

    if row.empty:
        return []

    symptom_row = row.drop(columns=["disease"]).iloc[0]
    return symptom_row[symptom_row == 1].index.tolist()


def get_questions_for_symptoms(symptom_ids):
    return symptom_questions[
        symptom_questions["symptom_id"].isin(symptom_ids)
    ]


def prioritize_questions(questions_df):
    merged = questions_df.merge(
        question_priority,
        on="symptom_id",
        how="left"
    )
    return merged.sort_values(
        by="priority_weight",
        ascending=False
    )

# ------------------------------------------------------------
# TRUE BAYESIAN UPDATE (LIKELIHOOD 0.85)
# ------------------------------------------------------------
def bayesian_update(prior_probs, symptom_id, answer):

    posterior = []

    for i, cls_code in enumerate(CLASS_CODES):

        cls_name = CLASS_LABELS[cls_code]

        row = disease_symptom_matrix[
            disease_symptom_matrix["disease"] == cls_name
        ]

        if row.empty:
            likelihood = 0.5
        else:
            symptom_present = row[symptom_id].values[0]

            if answer.lower() == "yes":
                likelihood = 0.85 if symptom_present == 1 else 0.15
            else:
                likelihood = 0.15 if symptom_present == 1 else 0.85

        posterior.append(prior_probs[i] * likelihood)

    posterior = np.array(posterior)
    posterior = posterior / posterior.sum()

    return posterior.tolist()

# ------------------------------------------------------------
# ADAPTIVE QUESTIONING (NOW 5 QUESTIONS)
# ------------------------------------------------------------
def adaptive_questioning_pipeline(image_output, max_questions=5):

    if image_output["decision"] != "ASK_MORE_QUESTIONS":
        return None

    probs = image_output["probabilities"]

    for _ in range(max_questions):

        predicted_disease = image_output["predicted_disease"]

        symptom_ids = get_symptoms_for_disease(predicted_disease)
        questions = get_questions_for_symptoms(symptom_ids)
        ordered_questions = prioritize_questions(questions)

        if ordered_questions.empty:
            break

        row = ordered_questions.iloc[0]
        symptom_id = row["symptom_id"]
        question = row["question"]

        print("\nQ:", question)

        # SIMULATION (replace with real input later)
        answer = "yes"
        print("A:", answer)

        probs = bayesian_update(probs, symptom_id, answer)

        confidence = max(probs)
        print("Updated confidence:", round(confidence, 4))

        if confidence >= HIGH_CONF_THRESHOLD:
            break

    final_index = probs.index(max(probs))

    return {
        "updated_prediction": CLASS_LABELS[CLASS_CODES[final_index]],
        "updated_confidence": max(probs),
        "updated_probabilities": probs
    }

# ------------------------------------------------------------
# FINAL DECISION
# ------------------------------------------------------------
def final_decision(image_output, reasoning_output=None):

    if reasoning_output is None:
        final_prediction = image_output["predicted_disease"]
        final_confidence = image_output["confidence"]
    else:
        final_prediction = reasoning_output["updated_prediction"]
        final_confidence = reasoning_output["updated_confidence"]

    if final_confidence >= HIGH_CONF_THRESHOLD:
        decision = "SAFE_ADVISORY"
    elif final_confidence >= LOW_CONF_THRESHOLD:
        decision = "LOW_CONFIDENCE_WARNING"
    else:
        decision = "ETHICAL_REFUSAL"

    return {
        "final_prediction": final_prediction,
        "final_confidence": round(final_confidence, 4),
        "decision": decision
    }

# ------------------------------------------------------------
# FULL PIPELINE
# ------------------------------------------------------------
def full_pre_diagnosis(image_path):

    image_output = image_pre_diagnosis(image_path)

    print("\nInitial prediction:", image_output["predicted_disease"])
    print("Initial confidence:", round(image_output["confidence"], 4))

    reasoning_output = adaptive_questioning_pipeline(image_output)

    final_output = final_decision(image_output, reasoning_output)

    return final_output

# ------------------------------------------------------------
# TEST RUN
# ------------------------------------------------------------
test_image_path = os.path.join(
    DATA_DIR,
    "HAM10000_images_part_1",
    os.listdir(os.path.join(DATA_DIR, "HAM10000_images_part_1"))[0]
)

final_result = full_pre_diagnosis(test_image_path)
final_result

Model exists: True
Using device: cpu
EfficientNet loaded successfully.
Medical datasets loaded.

Initial prediction: Melanocytic Nevus
Initial confidence: 0.7824


{'final_prediction': 'Melanocytic Nevus',
 'final_confidence': 0.7824,
 'decision': 'SAFE_ADVISORY'}